# Week 2 Day 2 Drill: Tone Rewriter

Build a Gradio app with Gemini:

1. **Inputs**: a textbox (the text to rewrite) and a dropdown with `"formal"`, `"funny"`, `"pirate"` (default: `"formal"`)
2. **Output**: a Markdown box
3. **Function** `rewrite(text, tone)`:
   - system prompt tells the model to rewrite the user's text in the chosen tone (use an f-string to put `tone` into it)
   - stream the answer from Gemini
   - `yield` the accumulated text
4. Launch it with `gr.Interface`

**Check yourself:** does your function take arguments in the same order as `inputs=[...]`? Are you yielding the full text so far?


In [7]:
import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI


In [9]:
load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL")
model = "gemini-3.1-flash-lite"

gemini = OpenAI(
    base_url=gemini_base_url,
    api_key=gemini_api_key
)
system_message = "You are a patient Python & LLM-engineering tutor who explains code line by line."


In [12]:
def callgemini(prompt):
    messages = [
        {"role" : "system", "content" : system_message},
        {"role" : "user", "content" : prompt}
    ]
    response = gemini.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content
        yield result



In [13]:
message_inputs = [
 gr.Textbox(),
 gr.Dropdown(
     choices=["formal", "funny", "pirate"],
     value="formal"
 )
]
message_output = gr.Markdown(label="Gemini")

view = gr.interface(
    fn=callgemini,
    inputs=[message_inputs],
    outputs=[message_output]
    )
view.launch()


TypeError: 'module' object is not callable